# EmpowerLens — Cascade runner (single backbone, disk-safe, timeout-safe, resumable)

**Fourth revision.** History: run 1 died on disk space (18 configs, no cleanup). Run 2
hung 11.4 hours inside one `evaluate.py` call and hit Kaggle's 12-hour limit with nothing
saved. Run 3 died at Stage 2 with `NameError: name 'STAGE2_SPLITS' is not defined` — a
kernel restart had wiped every variable defined in earlier cells, so the cascade
evaluation never ran at all.

### What this revision fixes

1. **Restart-proof state.** Config and helpers now live in
   `notebooks/cascade_bootstrap.py` on disk, not in kernel memory. Every code cell begins
   with `exec(open(BOOT).read())`, so any cell can be run on its own, in any order, after
   any restart. This is what kills the `NameError` — and the `os.chdir` it does also
   restores the working directory, which a restart resets too.
2. **Runs on `data/splits`, NOT `data/splits_combined`.** The combined dir has ~75%
   train/test leakage: CODIPAS overlaps `Annotated_data.csv` by 74% (1,937 of its 2,621
   rows), so merging pushed 194/253 val rows and 189/253 test rows into train. Everything
   in `results_combined/` is invalid. `data/splits` is verified clean — train-in-val 0,
   train-in-test 0. **Do not point `PARENT_SPLITS` back at the combined dir** until
   CODIPAS is deduplicated against the frozen Annotated val/test.
3. **Guard against the isolated-Stage-2 trap.** `src/evaluate.py` now refuses a
   distorted-only splits dir unless `--allow-distorted-only` is passed, and tags any such
   output `[stage2-isolated]`. Those numbers are diagnostics, never cascade results.
4. **Faster.** `--max-length 256 --truncation head_tail --batch-size 32` (token lengths
   are skewed — median 138, p95 482 — so the old 512 default was mostly padding), plus
   length-grouped batching wired into `WeightedTrainer` since transformers 5.x removed
   the `group_by_length` flag.

Carried over from earlier revisions: single-GPU pinning, hard per-subprocess timeouts
with process-group kills, per-seed syncing to `/kaggle/working/`, and skip-if-already-done
so a restart never retrains a finished config.

One backbone only — `mental/mental-roberta-base` — which beat DeBERTa-v3-base on every
metric in the prior side-by-side and had a much smaller val→test overfitting gap.

**Before running:**
1. Settings → **Accelerator: GPU** (prefer T4 — it has fp16 tensor cores), **Internet: On**.
2. `HF_TOKEN` Kaggle secret set — `mental-roberta-base` is gated, and you must also accept
   its licence on the Hub while logged in, or the token still 401s.
3. These pushed to the branch in cell 1: `notebooks/cascade_bootstrap.py`, `src/losses.py`,
   `src/make_splits_cascade.py`, `src/evaluate_cascade.py`, `src/evaluate.py`,
   `src/train_transformer.py`, and `data/splits/` + `data/splits_stage2/`.

**Run cells 1-5, 7, 11, 13.** Cell 9 (multiclass) is a comparison track, not part of the
cascade — skipping it saves about a third of the total training time.

**If the session dies partway:** re-run from the top. Every completed seed is detected via
its eval JSON and skipped instantly; you only lose the config that was mid-flight.

In [1]:
import os

# 1. Safely reset the working directory to the Kaggle root
os.chdir('/kaggle/working/')
!rm -rf /kaggle/working/empowerlens  # <-- ONLY delete the code repo, not your checkpoints!

# 2. Clone the repo from YOUR newly merged branch
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "nayab-space"   # must be a branch that HAS notebooks/cascade_bootstrap.py
                           # pushed to GitHub — Kaggle clones from the remote, so
                           # local-only commits are invisible here.

!git clone --branch $BRANCH $REPO_URL empowerlens
os.chdir('/kaggle/working/empowerlens')

# 3. Install the transformer stack AND captum
!pip install --upgrade pip setuptools wheel
!pip install -q -r requirements-transformer.txt
!pip install -q sentencepiece protobuf captum

Cloning into 'empowerlens'...
remote: Enumerating objects: 381, done.
remote: Counting objects: 100% (381/381), done.
remote: Compressing objects: 100% (279/279), done.
remote: Total 381 (delta 172), reused 303 (delta 94), pack-reused 0 (from 0)
Receiving objects: 100% (381/381), 7.94 MiB | 19.27 MiB/s, done.
Resolving deltas: 100% (172/172), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 41.6 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build depende

In [2]:
# 1a. mental/mental-roberta-base is GATED on the Hub — accept its terms at
#     https://huggingface.co/mental/mental-roberta-base while logged in, then add a
#     Kaggle Secret named HF_TOKEN (Add-ons -> Secrets) with a read-scope HF token.
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in to Hugging Face Hub.")
except Exception as e:
    print(f"[warn] no working HF_TOKEN secret ({e}) — training will 401 until this is set.")

Logged in to Hugging Face Hub.


In [3]:
# 1a2. Pin to a SINGLE GPU. Kaggle sometimes assigns T4 x2 -- transformers/accelerate's
#      weight-loading path can try to coordinate across all visible CUDA devices during
#      from_pretrained() even though nothing here intentionally uses 2 GPUs, and on some
#      driver/container combos that coordination deadlocks silently (this is the leading
#      suspect for the 11-hour hang seen at seed 1337's evaluate.py call last run).
# MUST run before torch/transformers are imported by any subprocess -- os.environ set here
# propagates to every `python -m src....` subprocess.run() call below since they inherit
# this process's environment.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-590dc8ce-4b88-7317-ba54-f77f17a6c0b2)
GPU 1: Tesla T4 (UUID: GPU-d034b969-a593-4f19-e908-0be791c63d5c)


In [4]:
# 1c. Load shared config + helpers from notebooks/cascade_bootstrap.py.
#
# These used to be defined inline. Kernel restarts wiped them, and running a
# later cell then died with `NameError: name 'STAGE2_SPLITS' is not defined`
# — which is exactly what killed Stage 2 (and therefore the cascade eval) on
# the last real run. Keeping the state on disk makes it survive restarts.
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())


In [5]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# Step 1 — derive Stage 2's distorted-only splits from PARENT_SPLITS.
#
# PARENT_SPLITS comes from the bootstrap and is data/splits (verified clean).
# It is deliberately NOT hardcoded here: this cell used to set
#   COMBINED_SPLITS = 'data/splits_combined'
# which silently overrode the bootstrap and rebuilt Stage 2 from the LEAKED
# dir (194/253 val and 189/253 test rows also present in train, because
# CODIPAS overlaps Annotated_data by 74%).

!python -m src.make_splits_cascade --source $PARENT_SPLITS --out $STAGE2_SPLITS --force
print(f"[step 1] Stage 2 splits derived from {PARENT_SPLITS}")


{
  "created_utc": "2026-08-14T18:43:02.986395+00:00",
  "purpose": "Stage 2 splits = --source splits filtered to y_bin==1 (distorted only). No re-splitting: row-level train/val/test boundaries are inherited unchanged from --source.",
  "source_dir": "data/splits_combined",
  "n_per_split": {
    "train": 2694,
    "val": 158,
    "test": 161
  },
  "n_removed_no_distortion": {
    "train": 1951,
    "val": 95,
    "test": 92
  },
  "per_class_train": {
    "emotional_reasoning": 347,
    "overgeneralization": 431,
    "mental_filter": 240,
    "should_statements": 324,
    "all_or_nothing": 244,
    "mind_reading": 465,
    "fortune_telling": 390,
    "magnification": 336,
    "personalization": 254,
    "labeling": 487
  },
  "source_file_sha256": {
    "train": "ef6eedbc773ff3965654a317b786b37a32fae23225b8968028a18f4782883357",
    "val": "ca10cf0b7a12f6aa642a927b3e4b26ddca55b51eff627a0028cb72f941c21f24",
    "test": "e8b757c52be9b971d27ff95e4611c358b5c72af9425b16b8af1e7069c0c47e93"

## Step 2 — Stage 1: binary model, 3 seeds

Trained on the **full** `PARENT_SPLITS` (`data/splits`) — it needs the No-Distortion rows,
since telling distorted from not-distorted is its entire job.

**`positive_class_f1` here is the ceiling on the whole cascade.** Any distorted row Stage 1
misses never reaches Stage 2 and is permanently wrong. If this number is low, the cascade
cannot beat a flat model no matter how good Stage 2 is — and that is the finding to
report, not a bug to chase.

Results print inline per seed and sync to `/kaggle/working/results_stage1/` immediately.

In [7]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# STAGE1_OUT comes from the bootstrap.
!mkdir -p $STAGE1_OUT

print(f"=== Stage 1 binary: {MODEL} ===")
for seed in SEEDS:
    # Use only the flags supported by your train_transformer.py parser
    extra_args = "--max-length 256 --truncation head_tail --batch-size 32 --epochs 3 --early-stopping-patience 2"
    run_and_report("binary", PARENT_SPLITS, STAGE1_OUT, seed, extra_flags=extra_args)

sync(STAGE1_OUT)
!df -h /kaggle/working

=== Stage 1 binary: mental/mental-roberta-base ===
$ CUDA_VISIBLE_DEVICES=0 python -m src.train_transformer --task binary --model mental/mental-roberta-base --seed 42 --device auto --splits data/splits_combined --batch-size 32 --epochs 3
[binary] device=cuda model=mental/mental-roberta-base train=4645 val=253 epochs=3


KeyboardInterrupt: 

## Step 2b — Multiclass (11-class), 3 seeds — comparison track, NOT part of the cascade

**Skippable.** Nothing downstream depends on it; skipping saves roughly a third of the
run. Included only to compare the cascade against a flat 11-class model.

Same data as Step 2. One fix over the earlier `results_combined` run:
`metric_for_best_model` now selects on `macro_f1_10` rather than `macro_f1`. Written to
`results_multiclass_v2/` so older numbers aren't overwritten.

Note the earlier `results_combined` numbers are not a fair baseline anyway — they were
trained on the leaked combined splits (see the header).

In [10]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# MULTICLASS_OUT comes from the bootstrap.
!mkdir -p $MULTICLASS_OUT

print(f"=== Multiclass (11-class): {MODEL} ===")
for seed in SEEDS:
    run_and_report(
        "multiclass", PARENT_SPLITS, MULTICLASS_OUT, seed,
        extra_flags="--max-length 256 --truncation head_tail --batch-size 32 --label-smoothing 0.1 --lr-scheduler cosine --early-stopping-patience 2",
    )

sync(MULTICLASS_OUT)
!df -h /kaggle/working

=== Multiclass (11-class): mental/mental-roberta-base ===
$ CUDA_VISIBLE_DEVICES=0 python -m src.train_transformer --task multiclass --model mental/mental-roberta-base --seed 42 --device auto --splits data/splits_combined --label-smoothing 0.1 --lr-scheduler cosine --early-stopping-patience 2
[multiclass] device=cuda model=mental/mental-roberta-base train=4645 val=253 epochs=4


KeyboardInterrupt: 

## Step 3 — Stage 2: multilabel head, distorted-only, 3 seeds

Trained on `data/splits_stage2` — derived from `PARENT_SPLITS` by keeping only `y_bin == 1`
rows. **train 1,278 / val 158 / test 161.**

This is a filter, not a re-split: every row keeps the train/val/test assignment it already
had, so no leakage is introduced. (If you see train ≈ 2,694 in Step 1's output, it was
built from the leaked combined dir — re-run Step 1.)

Stage 2 never sees `no_distortion`, so its capacity goes entirely on telling the ten types
apart. Uses focal loss + layer-wise LR decay for the minority classes (`all_or_nothing`,
`mental_filter`, `personalization`).

Its eval passes `--allow-distorted-only` because `src/evaluate.py` now blocks that dir by
default. Those numbers are **isolated diagnostics** — "how good is Stage 2 at its own
job?" — and are tagged `[stage2-isolated]`. They are not comparable to a flat model.

In [4]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# STAGE2_OUT comes from the bootstrap.
!mkdir -p $STAGE2_OUT

print(f"=== Stage 2 multilabel: {MODEL} (focal + LLRD) ===")
for seed in SEEDS:
    run_and_report(
        "multilabel", STAGE2_SPLITS, STAGE2_OUT, seed,
        extra_flags=(
            "--max-length 256 --truncation head_tail --batch-size 32 --loss focal --focal-gamma 2.0 --llrd --llrd-decay 0.9 --lr 3e-5 "
            "--lr-scheduler cosine --early-stopping-patience 2"
        ),
        # Stage 2 is evaluated on distorted-only splits, which src/evaluate.py
        # now refuses by default. These are ISOLATED diagnostics ("how good is
        # Stage 2 at its own job?") and are NOT cascade results — outputs are
        # tagged [stage2-isolated]. The honest number comes from Step 4 below.
        eval_flags="--allow-distorted-only",
    )

sync(STAGE2_OUT)
!df -h /kaggle/working

=== Stage 2 multilabel: mental/mental-roberta-base (focal + LLRD) ===


NameError: name 'STAGE2_SPLITS' is not defined

## Step 4 — end-to-end cascade evaluation (the number that counts)

`results_stage2/` above scores Stage 2 in isolation on distorted-only inputs. That looks
far better than reality for two reasons: the 92 No-Distortion rows have been removed from
the test set (36% of it, and the hardest part), and every false negative Stage 1 makes is
invisible.

This step chains Stage 1 → Stage 2 and scores the **composed** prediction against the FULL
val/test set. Rows Stage 1 rejects get an all-zero prediction and are scored against their
real labels like everything else, so Stage 1's errors are baked into the number.

**This is the only figure comparable to a flat multilabel model, and the only one to
report as "the cascade result".**

In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# CASCADE_OUT comes from the bootstrap.
!mkdir -p $CASCADE_OUT

print(f"=== Cascade eval: {TAG} (Stage 1 + Stage 2, matched seeds) ===")
for seed in SEEDS:
    stage1_ckpt = f"{CKPT_DIR}/binary_{TAG}_{seed}"
    stage2_ckpt = f"{CKPT_DIR}/multilabel_{TAG}_{seed}"
    sh(
        f"python -m src.evaluate_cascade --stage1-checkpoint {stage1_ckpt} "
        f"--stage2-checkpoint {stage2_ckpt} --splits {PARENT_SPLITS} --out {CASCADE_OUT}"
    )

sync(CASCADE_OUT)
!df -h /kaggle/working

## Step 5 — compare: multilabel flat vs cascade, multiclass old vs fixed

Compare `results_cascade` (composed) against a flat multilabel baseline.

⚠️ `results_combined/` is included below for reference only — those models were trained on
the leaked combined splits and saw ~75% of their own test set, so they are **not** a valid
baseline. A genuine comparison needs a flat multilabel model trained on `data/splits`.

In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

import pandas as pd

SOURCES = {
    "results_combined (flat, old)": "results_combined",
    "results_cascade (composed)": CASCADE_OUT,
    "results_multiclass_v2 (fixed)": MULTICLASS_OUT,
}

frames = []
for label, folder in SOURCES.items():
    p = Path(folder) / "paper_comparison.csv"
    if p.exists():
        d = pd.read_csv(p)
        # Older paper_comparison.csv rows embed the seed in the cascade model name
        # (cascade[binary_..._42+multilabel_..._42]), which made every seed its own
        # groupby group so mean +/- std came out NaN. evaluate_cascade.py no longer
        # writes it that way; this strips it from any CSV produced before that fix.
        d["model"] = d["model"].str.replace(r"_\d+(?=[+\]])", "", regex=True)
        d["results_dir"] = label
        frames.append(d)
    else:
        print(f"[skip] {p} not found")

all_results = pd.concat(frames, ignore_index=True)

view_ml = all_results[(all_results["task"] == "multilabel") & (all_results["split"] == "test")]
print("=== multilabel: flat vs cascade (test) ===")
print(view_ml.groupby(["results_dir", "model"])[["weighted_f1", "macro_f1"]].agg(["mean", "std"]).round(3))

view_mc = all_results[(all_results["task"] == "multiclass") & (all_results["split"] == "test")]
print("\n=== multiclass: old vs fixed (test) ===")
print(view_mc.groupby(["results_dir", "model"])[["weighted_f1", "macro_f1_10"]].agg(["mean", "std"]).round(3))

all_results.to_csv(f"{CASCADE_OUT}/flat_vs_cascade_vs_multiclass_comparison.csv", index=False)
sync(CASCADE_OUT)
print(f"\nWrote and synced flat_vs_cascade_vs_multiclass_comparison.csv")

In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# Final safety net — everything above already synced incrementally after each stage,
# this just re-confirms all four folders are present in /kaggle/working/.
for folder in (STAGE1_OUT, MULTICLASS_OUT, STAGE2_OUT, CASCADE_OUT):
    sync(folder)
!ls -la /kaggle/working
!df -h /kaggle/working

In [14]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

from pathlib import Path

for folder in ["results_stage1", "results_multiclass_v2", "results_stage2", "results_cascade"]:
    p = Path(f"/kaggle/working/{folder}")
    if p.exists():
        files = list(p.glob("*.json"))
        print(f"📁 {folder}: {len(files)} result files found")
        for f in files:
            print(f"   - {f.name}")
    else:
        print(f"📁 {folder}: Folder does not exist yet")

📁 results_stage1: Folder does not exist yet
📁 results_multiclass_v2: Folder does not exist yet
📁 results_stage2: 0 result files found
📁 results_cascade: Folder does not exist yet
